In [1]:
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer

# Load your dataset
df = pd.read_csv('trafficdata.csv')

# 1. Clean the 'violation_type' stringified lists
df['violation_type'] = df['violation_type'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

# 2. Flatten into binary columns
mlb = MultiLabelBinarizer()
violation_encoded = pd.DataFrame(mlb.fit_transform(df['violation_type']), columns=mlb.classes_, index=df.index)

# 3. Join back to the main dataframe
df = df.join(violation_encoded)

print("Flattened columns:", violation_encoded.columns.tolist())

Flattened columns: ['2W/3W - USING MOBILE PHONE', 'AGAINST ONE WAY/NO ENTRY', 'CARRYING LENGHTY MATERIAL', 'DEFECTIVE NUMBER PLATE', 'DEMANDING EXCESS FARE', 'DOUBLE PARKING', 'FAIL TO USE SAFETY BELTS', 'H T V PROHIBITED', 'JUMPING TRAFFIC SIGNAL', 'NO PARKING', 'OBSTRUCTING DRIVER', 'OTHER - USING MOBILE PHONE', 'PARKING IN A MAIN ROAD', 'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC', 'PARKING NEAR ROAD CROSSING', 'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS', 'PARKING ON FOOTPATH', 'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE', 'PARKING OTHER THAN BUS STOP', 'REFUSE TO GO FOR HIRE', 'RIDER NOT WEARING HELMET', 'STOPING ON WHITE/STOP LINE', 'U TURN PROHIBITED', 'USING BLACK FILM/OTHER MATERIALS', 'VIOLATING LANE DISIPLINE', 'WITHOUT SIDE MIRROR', 'WRONG PARKING']


In [2]:
# Convert to datetime
df['created_datetime'] = pd.to_datetime(df['created_datetime'], format='mixed')

# Extract useful features
df['hour'] = df['created_datetime'].dt.hour
df['day_of_week'] = df['created_datetime'].dt.dayofweek # 0=Monday, 6=Sunday
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

In [3]:
# Define weights
weights = {
    'PARKING IN A MAIN ROAD': 3,
    'WRONG PARKING': 2,
    'NO PARKING': 2,
    'PARKING ON FOOTPATH': 3,
    'DOUBLE PARKING': 3
}

# Calculate Score for each row
def calculate_score(row):
    score = 0
    for violation, weight in weights.items():
        if violation in row.index and row[violation] == 1:
            score += weight
    return score

df['priority_score'] = df.apply(calculate_score, axis=1)

In [4]:
# Aggregate by location to see the worst areas
hotspots = df.groupby('junction_name')[['priority_score', 'id']].agg({
    'priority_score': 'sum',
    'id': 'count'
}).sort_values(by='priority_score', ascending=False)

print(hotspots.head(10))

                                  priority_score      id
junction_name                                           
No Junction                               371892  147880
BTP051 - Safina Plaza Junction             32708   15449
BTP082 - KR Market Junction                24134   11538
BTP044 - Sagar Theatre Junction            22402   10549
BTP040 - Elite Junction                    21827   10718
BTP211 - Central Street Junction           11697    5388
BTP058 - Subbanna Junction                 10609    5189
BTP027 - Modi Bridge Junction               9443    4584
BTP020 - Hosahalli Metro Station            8453    4101
BTP057 - Anand Rao Junction                 8368    3935


In [5]:
import sys; print(sys.executable)

/usr/local/bin/python3


In [6]:
from sklearn.cluster import DBSCAN
import numpy as np

coords = df[['latitude','longitude']].values
coords_rad = np.radians(coords)

db = DBSCAN(eps=0.3/6371, min_samples=10, algorithm='ball_tree', metric='haversine')
df['cluster'] = db.fit_predict(coords_rad)

# Then rank clusters by violation count + priority score
cluster_summary = df[df['cluster'] != -1].groupby('cluster').agg(
    violation_count=('id','count'),
    avg_priority=('priority_score','mean'),
    dominant_type=('violation_type', lambda x: x.mode()[0])
).sort_values('violation_count', ascending=False)

In [5]:
# ...existing code...
import folium
from folium.plugins import HeatMap

def find_lat_lon(df):
    cols = {c.lower().strip(): c for c in df.columns}
    lat_keys = ['latitude','lat']
    lon_keys = ['longitude','lon','lng']
    lat_col = next((cols[k] for k in lat_keys if k in cols), None)
    lon_col = next((cols[k] for k in lon_keys if k in cols), None)
    return lat_col, lon_col

lat_col, lon_col = find_lat_lon(df)
if lat_col is None or lon_col is None:
    raise ValueError(f"Latitude/longitude columns not found. Candidates: {list(df.columns)}")

# coerce to numeric and warn about dropped rows
df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
if 'priority_score' not in df.columns:
    df['priority_score'] = 0
else:
    df['priority_score'] = pd.to_numeric(df['priority_score'], errors='coerce').fillna(0)

map_df = df.dropna(subset=[lat_col, lon_col]).copy()
print(f"Rows with valid coords: {len(map_df)}")

if map_df.empty:
    raise ValueError("No rows with valid latitude/longitude after coercion.")

# Build heat data as [lat, lon, weight] (folium expects [lat, lon, weight])
heat_data = map_df[[lat_col, lon_col, 'priority_score']].values.tolist()

# Normalize weights a bit (optional) to avoid zero intensity everywhere
weights = map_df['priority_score'].values
if weights.max() > 0:
    map_df['heat_weight'] = map_df['priority_score'] / max(weights)
    heat_data = map_df[[lat_col, lon_col, 'heat_weight']].values.tolist()
else:
    # keep zeros allowed; folium will still show points
    heat_data = map_df[[lat_col, lon_col, 'priority_score']].values.tolist()

# Create map centered on mean coords
m = folium.Map(location=[map_df[lat_col].mean(), map_df[lon_col].mean()], zoom_start=13)

# Add heatmap; set min_opacity and max_zoom for better visibility
HeatMap(heat_data, radius=10, blur=15, min_opacity=0.2, max_zoom=18).add_to(m)

out_path = "enforcement_hotspots.html"
m.save(out_path)
print(f"Map saved to {out_path} ({len(heat_data)} points).")
# ...existing code...

Rows with valid coords: 298450
Map saved to enforcement_hotspots.html (298450 points).


In [7]:
# 1. Cluster centroids (missing from your code)
cluster_centers = df[df['cluster'] != -1].groupby('cluster').agg(
    center_lat=('latitude', 'mean'),
    center_lon=('longitude', 'mean'),
    violation_count=('id', 'count'),
    avg_priority=('priority_score', 'mean'),
    peak_hour=('hour', lambda x: x.mode()[0]),
    peak_day=('day_of_week', lambda x: x.mode()[0]),
    top_junction=('junction_name', lambda x: x.mode()[0])
).reset_index()

# 2. is_resolved flag (was planned, never built)
df['is_resolved'] = df['action_taken_timestamp'].notna().astype(int)
# → 100% zeros with current data, but shows analytical thinking

# 3. Composite risk score per cluster (upgrade from raw priority)
cluster_centers['risk_score'] = (
    0.4 * (cluster_centers['violation_count'] / cluster_centers['violation_count'].max()) +
    0.3 * (cluster_centers['avg_priority'] / cluster_centers['avg_priority'].max()) +
    0.3 * cluster_centers['peak_hour'].apply(lambda h: 1 if 7<=h<=10 or 17<=h<=20 else 0.5)
) * 10  # scale to 0-10

# 4. Export
cluster_centers.to_csv('cluster_data.csv', index=False)

In [9]:
import pandas as pd
cluster_df = pd.read_csv("cluster_data.csv")
hotspot_df = pd.read_csv("hotspot_report.csv")

print("=== RISK SCORE STATS ===")
print(cluster_df["risk_score"].describe())

print("\n=== TIER DISTRIBUTION ===")
print(hotspot_df["risk_tier"].value_counts())

print("\n=== SCORE DISTRIBUTION (buckets) ===")
print(pd.cut(cluster_df["risk_score"], bins=[0,2,4,7,10]).value_counts().sort_index())

print("\n=== COMPONENT STATS ===")
print("violation_count:", cluster_df["violation_count"].describe())
print("\navg_priority:", cluster_df["avg_priority"].describe())
print("\npeak_hour_frac:", cluster_df["peak_hour_frac"].describe())
print("\njunc_flag dist:", (cluster_df["top_junction"].apply(lambda j: 0 if str(j).strip().lower() == "no junction" else 1)).value_counts())


=== RISK SCORE STATS ===
count    869.000000
mean       0.927031
std        0.733078
min        0.000000
25%        0.310000
50%        0.920000
75%        1.330000
max        5.560000
Name: risk_score, dtype: float64

=== TIER DISTRIBUTION ===
risk_tier
🟢 LOW       801
🟡 MEDIUM     66
🟠 HIGH        2
Name: count, dtype: int64

=== SCORE DISTRIBUTION (buckets) ===
risk_score
(0, 2]     735
(2, 4]      65
(4, 7]       2
(7, 10]      0
Name: count, dtype: int64

=== COMPONENT STATS ===
violation_count: count      869.000000
mean       314.142693
std       2125.200386
min          7.000000
25%         21.000000
50%         37.000000
75%        112.000000
max      55527.000000
Name: violation_count, dtype: float64

avg_priority: count    869.000000
mean       2.800763
std        1.208692
min        2.000000
25%        2.096774
50%        2.304833
75%        2.910000
max       11.230769
Name: avg_priority, dtype: float64

peak_hour_frac: count    869.000000
mean       0.161846
std        0.